##### **** These pip installs need to be adapted to use the appropriate release level. Alternatively, The venv running the jupyter lab could be pre-configured with a requirement file that includes the right release. Example for transform developers working from git clone:
```
make venv 
source venv/bin/activate 
pip install jupyterlab
```

In [1]:
%%capture
!pip install pandas

In [2]:
import os
os.environ['S3_ACCESS_KEY']=os.environ['COS_DPK_ACCESS_KEY']
os.environ['S3_SECRET_KEY']=os.environ['COS_DPK_SECRET_KEY']
os.environ['S3_ENDPOINT']= "https://s3.us-south.cloud-object-storage.appdomain.cloud"
input_folder="bucket-wikimedia-1/take6_md"
output_folder="bucket-wikimedia-1/test-MT"

##### ***** Import required classes and modules

In [3]:
from data_processing.data_access import DataAccessS3

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
da=DataAccessS3(config={'input_folder': input_folder,'output_folder': output_folder})
l, _, _=da.get_files_to_process()

In [5]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

dataset=None
ndx = 0
for _filename in l:
    byte_array,_=da.get_file(_filename)
    ndx = ndx + 1
    print (f"{ndx}- Processing {_filename}: file size: {len(byte_array)}")
    pqfile=pq.ParquetFile(pa.BufferReader(byte_array))
    patable=pqfile.read().select(['filename', 'hash'])
    if dataset is None:
        dataset=patable.to_pandas()
    else:
        dataset=pd.concat([dataset, patable.to_pandas()], ignore_index=True)


1- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_0.ndjson.parquet: file size: 35053613
2- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_1.ndjson.parquet: file size: 35506589
3- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_10.ndjson.parquet: file size: 34491630
4- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_100.ndjson.parquet: file size: 50950761
5- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_101.ndjson.parquet: file size: 52202646
6- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_102.ndjson.parquet: file size: 52989262
7- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_103.ndjson.parquet: file size: 53085169
8- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_104.ndjson.parquet: file size: 53694756
9- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_105.ndjson.parquet: file size: 53821285
10- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_106.ndjson.parquet: file size: 531

In [6]:
hash_counts=dataset['hash'].value_counts()
dataset['hash_count']=dataset['hash'].map(hash_counts)
dataset.sort_values(by='hash_count', ascending=False)

,filename,hash,hash_count
5359647,enwiki_namespace_0_391_3838.html,06f9b993ca3e8324d22b152b154671f8e2637cb802a4de...,25
5322705,enwiki_namespace_0_389_5065.html,06f9b993ca3e8324d22b152b154671f8e2637cb802a4de...,25
5362312,enwiki_namespace_0_391_6503.html,06f9b993ca3e8324d22b152b154671f8e2637cb802a4de...,25
5305911,enwiki_namespace_0_387_547.html,06f9b993ca3e8324d22b152b154671f8e2637cb802a4de...,25
5310785,enwiki_namespace_0_387_5421.html,06f9b993ca3e8324d22b152b154671f8e2637cb802a4de...,25
...,...,...,...
2377012,enwiki_namespace_0_185_8250.html,a783df82d337d7e6c0ba476b8a14c0592742503967be17...,1
2377011,enwiki_namespace_0_185_8249.html,2c5e0189312dcb578f9a171b0624e327e989fc49d9e3d3...,1
2377010,enwiki_namespace_0_185_8248.html,30dd51f4bff99670e8e5dcedc462e81dd1e329cc160c97...,1
2377009,enwiki_namespace_0_185_8247.html,c7531ff32e4bdf6674a38376afeb0b4c4af2f5ab3e8466...,1


In [16]:
dataset.drop_duplicates(subset=['hash'])

,filename,hash,hash_count
0,enwiki_namespace_0_0_0.html,756337333d16a5817f98cd6893580a5bfc65609df400e2...,1
1,enwiki_namespace_0_0_1.html,bf001f7604b81e3ce37d8e3120fe9465bb79fce7d1a162...,1
2,enwiki_namespace_0_0_2.html,0f3afa4377b6f4b82fd7356cb3be31649ea709a89f906c...,1
3,enwiki_namespace_0_0_3.html,90d6286d3e8f0c109cd5179badc36a74741d2bc53bf344...,1
4,enwiki_namespace_0_0_4.html,0b91160be7fc42e994d122fff3398f1b9a5980e5cf4248...,1
...,...,...,...
7128268,enwiki_namespace_0_99_24186.html,9fc2b14d324a7f5ea10d572a423dcc66203d94081da751...,1
7128269,enwiki_namespace_0_99_24187.html,be51be07000955d5572ebf664452a734f30909ca93f2c2...,1
7128270,enwiki_namespace_0_99_24188.html,a8af1432c181d6bacab82eb4e49df81637d050206db234...,1
7128271,enwiki_namespace_0_99_24189.html,73ab125dbe590b3c8bd7987e44a07544b2b52dd65b2eb7...,1


In [7]:
dataset.drop_duplicates(subset=['hash'])

,filename,hash,hash_count
0,enwiki_namespace_0_0_0.html,756337333d16a5817f98cd6893580a5bfc65609df400e2...,1
1,enwiki_namespace_0_0_1.html,bf001f7604b81e3ce37d8e3120fe9465bb79fce7d1a162...,1
2,enwiki_namespace_0_0_2.html,0f3afa4377b6f4b82fd7356cb3be31649ea709a89f906c...,1
3,enwiki_namespace_0_0_3.html,90d6286d3e8f0c109cd5179badc36a74741d2bc53bf344...,1
4,enwiki_namespace_0_0_4.html,0b91160be7fc42e994d122fff3398f1b9a5980e5cf4248...,1
...,...,...,...
7128268,enwiki_namespace_0_99_24186.html,9fc2b14d324a7f5ea10d572a423dcc66203d94081da751...,1
7128269,enwiki_namespace_0_99_24187.html,be51be07000955d5572ebf664452a734f30909ca93f2c2...,1
7128270,enwiki_namespace_0_99_24188.html,a8af1432c181d6bacab82eb4e49df81637d050206db234...,1
7128271,enwiki_namespace_0_99_24189.html,73ab125dbe590b3c8bd7987e44a07544b2b52dd65b2eb7...,1


In [17]:
byte_array,_=da.get_file(f"{input_folder}/enwiki_namespace_0_391.ndjson.parquet")
_hash=dataset.sort_values(by='hash_count', ascending=False)['hash'].iloc[0]
df=pq.ParquetFile(pa.BufferReader(byte_array)).read().to_pandas()
df[df['hash']==_hash][['filename','contents']]

,filename,contents
3405,enwiki_namespace_0_391_3414.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
3625,enwiki_namespace_0_391_3634.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
3829,enwiki_namespace_0_391_3838.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
4030,enwiki_namespace_0_391_4039.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
4278,enwiki_namespace_0_391_4287.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
6494,enwiki_namespace_0_391_6503.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
6836,enwiki_namespace_0_391_6845.html,# Bill Charlap\n\nAmerican jazz pianist (born ...
